# Étape 2 : nettoyage des textes

Minuscules, suppression des emojis et de la ponctuation, espaces normalisés. Le `_` est conservé : les balises (`personne_tag`, `ip_tag`...) restent intactes.
Entrée : `data/train_anonyme_2.csv`, `data/test_anonyme_2.csv` — Sortie : `data/train_clean.csv`, `data/test_clean.csv`.

In [ ]:
import re
import pandas as pd
from tqdm.auto import tqdm   # barre de progression

In [ ]:
# Emojis et pictogrammes (plages Unicode courantes : symboles, dingbats, drapeaux, émojis récents)
RE_EMOJI = re.compile(
    "[\U0001F000-\U0001FAFF"   # émojis, pictogrammes, drapeaux
    "\U00002600-\U000027BF"    # symboles divers et dingbats
    "\U00002B00-\U00002BFF"    # flèches et symboles divers
    "\u200d\ufe0f]+",          # liant d'émojis et sélecteur de variation
    flags=re.UNICODE)


def clean_text(text: str) -> str:
    """Nettoie un commentaire déjà anonymisé."""
    text = text.lower()                          # 1. minuscules
    text = RE_EMOJI.sub(" ", text)               # 2. retire les emojis
    text = re.sub(r"[^\w\s]", " ", text)         # 3. retire la ponctuation ("_" conservé -> balises intactes)
    return re.sub(r"\s+", " ", text).strip()     # 4. espaces multiples et retours à la ligne -> un espace

In [ ]:
# --- Application à la colonne comment_text + vérification des balises ---
# Les balises ont la forme xxx_tag (personne_tag, ip_tag, url_tag...)
RE_BALISE = r"[a-z]+_tag"

FICHIERS_CLEAN = {
    "data/train_anonyme_2.csv": "data/train_clean.csv",
    "data/test_anonyme_2.csv": "data/test_clean.csv",
}

for entree, sortie in FICHIERS_CLEAN.items():
    df = pd.read_csv(entree)
    avant = df["comment_text"].fillna("")
    apres = pd.Series([clean_text(t) for t in tqdm(avant, desc=entree)], index=avant.index)

    df["comment_text"] = apres
    df.to_csv(sortie, index=False)
    print(f"\n===== {sortie} ({len(df)} lignes) =====")

    # 5 exemples avant/après (de préférence des commentaires contenant une balise)
    avec_balise = avant[avant.str.contains("_tag")]
    for i in (avec_balise if len(avec_balise) >= 5 else avant).sample(5, random_state=0).index:
        print("AVANT :", avant[i][:200].replace("\n", " "))
        print("APRES :", apres[i][:200], "\n")

    # Vérification : le nombre de balises *_tag doit être identique, ligne par ligne, avant et après
    assert (avant.str.count(RE_BALISE) == apres.str.count(RE_BALISE)).all(), "une balise *_tag a été cassée"
    print("Vérification OK : aucune balise *_tag n'a été cassée.")